# RoBERTa-base fine-tune for News Bias Classifier (Colab)

Two trainings in one run: **media-split** (outlet-disjoint, the headline number) and **random-split** (sanity, expected to be much higher). The contrast between the two is the writeup's central finding.

Runtime: **~50–60 min on a free T4 GPU**. Set `Runtime > Change runtime type > T4 GPU` first.

Inputs uploaded from local `data/processed/`:
- `train.parquet`, `val.parquet`, `test.parquet` (media split)
- `train_random.parquet`, `val_random.parquet`, `test_random.parquet` (random split)

Outputs (auto-downloaded):
- `roberta_media.zip`
- `roberta_random.zip`

Unzip both into your local project root so you have `models/roberta_media/` and `models/roberta_random/`.

## 1. GPU check

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'No GPU — Runtime > Change runtime type > T4 GPU'

## 2. Install pinned deps

In [ ]:
!pip install -q 'transformers>=4.40' 'datasets>=2.18' 'accelerate>=0.29' 'scikit-learn>=1.4' pyarrow

## 3. Upload all six parquets
Multi-select the three media-split files (`train.parquet`, `val.parquet`, `test.parquet`) AND the three random-split files (`train_random.parquet`, etc.) from your local `data/processed/` folder. Six files total.

In [ ]:
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    print(name, len(uploaded[name]) // 1024, 'KB')
required = ['train.parquet', 'val.parquet', 'test.parquet',
            'train_random.parquet', 'val_random.parquet', 'test_random.parquet']
missing = [r for r in required if r not in uploaded]
assert not missing, f'Missing files: {missing} — re-run this cell and select all six.'

## 4. Shared helpers (cleaner, head+tail truncation, training loop)

In [ ]:
import os, re, shutil
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    DataCollatorWithPadding, EarlyStoppingCallback, Trainer, TrainingArguments,
)

MODEL_NAME = 'roberta-base'
MAX_LENGTH = 512
EPOCHS = 2
BATCH_SIZE = 16
LR = 1e-5
WEIGHT_DECAY = 0.1
WARMUP_RATIO = 0.1

LABELS = ['left', 'center', 'right']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

# --- Cleaner: must match src/data/clean.py exactly so train/inference agree.
OUTLET_NAMES = {
    'cnn','msnbc','fox','foxnews','fox news','npr','monitor','csmonitor',
    'reuters','ap','associated press','bloomberg','bbc','afp',
    'nyt','new york times','wsj','wall street journal',
    'washingtonpost','washington post','wapo',
    'politico','axios','the hill','atlantic','slate','salon',
    'huffpost','huffington post','vox','intercept','buzzfeed','vice',
    'breitbart','newsmax','dailywire','daily wire','daily caller',
    'the federalist','national review','thehill','guardian','telegraph',
    'usa today','usatoday',
}
_OUTLET = re.compile(r'\b(?:' + '|'.join(sorted(OUTLET_NAMES, key=len, reverse=True)) + r')\b', re.IGNORECASE)
_BOILERPLATE = re.compile('|'.join([
    r'story highlights?', r'replay\s+more\s+videos?\s+must\s+watch',
    r'more\s+videos?\s+must\s+watch', r'more\s+videos?', r'must\s+watch',
    r'just\s+watched', r'\breplay\b',
    r'delivered\s+to\s+(?:your\s+)?inbox',
    r'sign\s+up\s+for\s+(?:our\s+)?newsletter',
    r'subscribe\s+to\s+(?:our\s+)?newsletter',
    r'privacy\s+policy', r'by\s+signing\s+up', r'you\s+agree\s+to\s+our',
    r'©\s*\d{4}.*?reserved', r'all\s+rights\s+reserved',
    r'click\s+here\s+to', r'follow\s+us\s+on\s+(?:twitter|facebook)',
    r'read\s+more\s*:', r'^\s*\(reuters\)\s*[-—]\s*',
]), re.IGNORECASE)
_WS = re.compile(r'\s+')

def clean_for_modeling(text):
    if not text: return ''
    s = _BOILERPLATE.sub(' ', text)
    s = _OUTLET.sub(' ', s)
    return _WS.sub(' ', s).strip().lower()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def head_tail_tokenize(texts, max_length=MAX_LENGTH):
    half = max_length // 2
    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id
    input_ids_batch, attention_batch = [], []
    for t in texts:
        ids = tokenizer(t, add_special_tokens=False, truncation=False)['input_ids']
        if len(ids) <= max_length - 2:
            chosen = ids
        else:
            chosen = ids[: half - 1] + ids[-(half - 1):]
        ids_full = [cls_id] + chosen + [sep_id]
        input_ids_batch.append(ids_full)
        attention_batch.append([1] * len(ids_full))
    return {'input_ids': input_ids_batch, 'attention_mask': attention_batch}

def tokenize_batch(batch):
    cleaned = [clean_for_modeling(t) for t in batch['text']]
    return head_tail_tokenize(cleaned)

def load(path):
    df = pd.read_parquet(path)
    return df[df['text'].str.len() > 0].reset_index(drop=True)

def to_hf(df):
    return Dataset.from_pandas(
        df[['text', 'label_id']].rename(columns={'label_id': 'labels'}),
        preserve_index=False,
    )

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds),
            'macro_f1': f1_score(labels, preds, average='macro')}

def train_one(split_name, train_path, val_path, test_path, output_dir):
    print(f'\n{"="*60}\nTraining {MODEL_NAME} on {split_name} split\n{"="*60}')
    train_df = load(train_path); val_df = load(val_path); test_df = load(test_path)
    print(f'train={len(train_df)} val={len(val_df)} test={len(test_df)}')

    train_ds = to_hf(train_df).map(tokenize_batch, batched=True, remove_columns=['text'])
    val_ds = to_hf(val_df).map(tokenize_batch, batched=True, remove_columns=['text'])
    test_ds = to_hf(test_df).map(tokenize_batch, batched=True, remove_columns=['text'])

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID,
    )

    args = TrainingArguments(
        output_dir=f'_runs_{split_name}',
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=LR,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        save_total_limit=1,
        logging_steps=50,
        fp16=True,
        report_to=[],
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )
    trainer.train()

    out = trainer.predict(test_ds)
    preds = np.argmax(out.predictions, axis=-1)
    print(f'\n=== test ({split_name}) ===')
    print(classification_report(out.label_ids, preds, target_names=LABELS, digits=4, zero_division=0))
    print('confusion matrix:')
    print(pd.DataFrame(
        confusion_matrix(out.label_ids, preds, labels=list(range(len(LABELS)))),
        index=LABELS, columns=LABELS,
    ))

    os.makedirs(output_dir, exist_ok=True)
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    archive = shutil.make_archive(f'roberta_{split_name}', 'zip', root_dir='.', base_dir=output_dir)
    print(f'Created {archive}')
    return archive

## 5. Train on the media (outlet-disjoint) split
~25 min. This is the hard split — the headline number for your writeup.

In [ ]:
media_zip = train_one('media', 'train.parquet', 'val.parquet', 'test.parquet',
                       'models/roberta_media')

## 6. Train on the random split
~25 min. Sanity check — expected to be much higher than media split. The *gap* between these two numbers is your writeup's main finding.

In [ ]:
random_zip = train_one('random', 'train_random.parquet', 'val_random.parquet', 'test_random.parquet',
                        'models/roberta_random')

## 7. Download both zips
Two files will save to your browser. Move them into your project root and unzip.

In [ ]:
from google.colab import files
files.download(media_zip)
files.download(random_zip)